In [14]:
#importing stuff
import numpy as np                                      
import pandas as pd
import matplotlib.pyplot as plt
import math,os,random,sys,h5py
from sklearn import preprocessing
from sklearn.svm import SVR
from sklearn.impute import SimpleImputer
from rdkit import Chem
from mordred import Calculator, descriptors
from sklearn.exceptions import ConvergenceWarning

In [15]:
APs = pd.read_csv("tripeptide_AP.txt", sep=": ", index_col=0, header=None, engine = 'python')       #AP scores from simulation
f = h5py.File('tripeptides.hdf5','r')                                                               #reading data from the file generated from judred generator
peps = np.array(f.get('peptides'))
feas = np.array(f.get('features'))
data = np.array(f.get('data'))
f.close()
judp = pd.DataFrame(data, columns = feas, index = peps)                                             #arranging it into a pandas data frame
smiles = pd.read_csv("smi.txt", sep=": ", engine='python')              #smiles data of peptides by their names
modp = pd.read_csv('Mordred_Parameters_to_be_used.csv', sep=',', engine='python')     #mordred data in same order as the smiles data

In [16]:
x_train_jd = [np.array(judp.loc[b'ALA-ALA-ALA'])]                  # initializing the training data with data of polyalanine 
x_train_md = [np.array(modp.iloc[1])]
Y_train = np.array(APs.loc['ALA-ALA-ALA'])
print(x_train_jd)
print(x_train_md)
print(Y_train)

[array([  0.     ,   0.     , 231.26942,   0.     ,  -0.99   ,   0.     ,
       387.     ,   0.     ,  34.5    ,   0.     ], dtype=float32)]
[array([ 1.00000000e+00,  7.46974883e+00,  3.91400279e+00,  1.91158910e-01,
        3.38139405e+00,  1.22163247e-01, -3.14179120e-02, -2.75043650e-02,
       -4.58309828e-01,  5.25622555e-01, -3.13695140e-02,  3.30213088e-01,
        3.13812757e+02, -1.03679109e+00, -9.07644031e-01, -7.41655577e-01,
       -1.07044306e+00,  3.14764474e-01,  4.92290242e+01,  8.76488202e-01,
        1.79239204e+00,  1.22655123e+00,  8.80867851e-01,  8.83171240e-01,
       -9.82029000e-03, -1.77908257e-01, -6.02702350e-02,  6.19418065e-01,
        8.21173730e-02,  2.56494936e+00,  1.18548349e+01,  6.33814029e-01,
        1.82948988e+00,  3.27677099e+00,  6.66666667e-01,  6.66666667e-01,
        1.00000000e+00,  0.00000000e+00,  3.75000000e-01,  0.00000000e+00,
        0.00000000e+00,  4.53389831e-01,  3.81355932e-01,  6.47933884e+00,
        1.13032269e+00,  0.00000

In [17]:
def randPepSelector(APs, judp, modp, smiles, x_train_jd, x_train_md, Y_train):                           # a is APs, b is judp/modp
    for pep in range(10):                                              # generates random indecies, looks up those indecies data and adds to the training data
        i = random.randrange(1,8000)
        p = APs.iloc[i]
        Y_train = np.append(Y_train, np.array(p), axis=0)
        x_train_jd = np.append(x_train_jd, [np.array(judp.loc[p.name.encode('utf-8')])],axis=0)
        x_train_md = np.append(x_train_md, np.array(modp.iloc[smiles.index[smiles['Peptides']==p.name]]), axis=0)
    return Y_train,x_train_jd, x_train_md  

In [18]:
Y_train,x_train_jd,x_train_md = randPepSelector(APs,judp,modp,smiles,x_train_jd,x_train_md,Y_train)                 # first we take a set of random peptides(10)
print(x_train_jd)
print(x_train_md)
print(Y_train)

[[ 0.0000000e+00  0.0000000e+00  2.3126942e+02  0.0000000e+00
  -9.9000001e-01  0.0000000e+00  3.8700000e+02  0.0000000e+00
   3.4500000e+01  0.0000000e+00]
 [ 2.0000000e+00  0.0000000e+00  3.6136942e+02  0.0000000e+00
  -4.1300001e+00 -2.0000000e+00  5.8700000e+02  3.3333334e-01
   4.4759998e+01  0.0000000e+00]
 [ 9.0000000e+00  0.0000000e+00  4.2246945e+02  1.0000000e+00
  -2.3900001e+00 -1.0000000e+00  6.4500000e+02  3.0000000e+00
   4.6809998e+01  0.0000000e+00]
 [ 1.0000000e+00  1.0000000e+00  3.2029944e+02  0.0000000e+00
  -8.5000002e-01  0.0000000e+00  5.3500000e+02  2.5000000e-01
   3.3389999e+01  2.0000000e+00]
 [ 1.1000000e+01  0.0000000e+00  4.7256943e+02  1.0000000e+00
   7.4000001e-01  0.0000000e+00  7.3300000e+02  2.2000000e+00
   5.1610001e+01  0.0000000e+00]
 [ 6.0000000e+00  0.0000000e+00  3.2335947e+02  0.0000000e+00
  -8.8999999e-01  0.0000000e+00  5.2100000e+02  2.0000000e+00
   4.1029999e+01  1.0000000e+00]
 [ 1.0000000e+00  1.0000000e+00  3.4638947e+02  0.0000000e

In [19]:
def loc(s,Y_pred):                      #takes the APs of top peptides and returns their location in judred parameters so we can look up their names by their location
    y = []
    for i in range(len(s)):
        for j in range(len(Y_pred)):
            if s[i] == Y_pred[j]:
                if j not in y:
                    y.append(j)
    return y

In [20]:
def topN(n,judp,x_train_jd,Y_train):                                                                #takes the judred parameters and the training data 
    svr = SVR(kernel='rbf',gamma='scale',C=100,epsilon=0.1,max_iter=-1,tol=0.0001,verbose=0)    #trains a svm(rbf)
    svr.fit(x_train_jd,Y_train)
    x = []
    for i in range(8000):
        x.append(np.array(judp.iloc[i]))
    imputer = SimpleImputer(strategy='mean')                                                    #imputer removes NaN values from the data set
    x_imputed = imputer.fit_transform(x)
    Y_pred = svr.predict(x_imputed)                                                             #predicts AP scores for all peptides
    s = sorted(Y_pred)
    s = s[8000-n:]
    y_loc = loc(s,Y_pred)
    y_nam = ['']*len(y_loc)
    for i in range(len(y_loc)):
        y_nam[i] = judp.iloc[y_loc[i]].name
    return y_nam,y_loc   

In [21]:
def addToTrain(y_nam,y_loc,APs,judp,modp,x_train_jd,x_train_md,Y_train):                                               #takes the predicted peptide in previous iteration and adds them to training data
    for i in y_nam:
        x_train_jd = np.append(x_train_jd, [np.array(judp.loc[i])], axis=0)
        x_train_md = np.append(x_train_md, np.array(modp.iloc[smiles.index[smiles['Peptides']==i.decode(encoding='utf-8')]]), axis=0)
        Y_train = np.append(Y_train, np.array(APs.loc[i.decode(encoding='utf-8')]), axis=0)
    return x_train_jd,x_train_md,Y_train

In [22]:
def topn(n,smiles, modp, y_nam_jd, x_train_md, Y_train):
    svr = SVR(kernel='rbf',gamma='scale',C=100,epsilon=0.1,max_iter=-1,tol=0.0001,verbose=0)    #trains a svm(rbf)
    svr.fit(x_train_md,Y_train)
    x = np.empty((0,modp.shape[1]))
    for i in y_nam_jd:
        x = np.append(x, np.array(modp.iloc[smiles.index[smiles['Peptides']==i.decode(encoding='utf-8')]]), axis=0)
    Y_pred = svr.predict(x)                                                             #predicts AP scores for all peptides
    pred = pd.DataFrame(data=Y_pred,index=y_nam_jd,columns=['prediction'])
    y_nam = np.array(pred['prediction'].nlargest(n=n).index)
    return y_nam


In [23]:
def runAL(num_iter,APs,judp,smiles,modp,x_train_jd,x_train_md,Y_train):
    for i in range(num_iter):
        y_nam_jd,y_loc = topN(math.ceil((np.log(3)**2) * 1000),judp,x_train_jd,Y_train)
        y_nam= topn(10,smiles, modp, y_nam_jd, x_train_md, Y_train)
        print(y_nam)
        x_train_jd,x_train_md,Y_train = addToTrain(y_nam,y_loc,APs,judp,modp,x_train_jd,x_train_md,Y_train)
    return topn(40,smiles, modp, y_nam_jd, x_train_md, Y_train)

y_nam = runAL(80,APs,judp,smiles,modp,x_train_jd,x_train_md,Y_train)
print(y_nam)

[b'TYR-GLY-GLY' b'TYR-GLY-CYS' b'TYR-CYS-GLY' b'TYR-GLY-ALA'
 b'TYR-ALA-GLY' b'TYR-SER-GLY' b'TRP-GLY-GLY' b'TYR-MET-MET'
 b'TYR-GLN-MET' b'TYR-ARG-CYS']
[b'VAL-CYS-GLY' b'VAL-GLY-CYS' b'TRP-GLY-GLY' b'TYR-GLY-GLY'
 b'THR-CYS-GLY' b'SER-CYS-GLY' b'TRP-ALA-GLY' b'THR-GLY-CYS'
 b'SER-GLY-CYS' b'TRP-CYS-GLY']
[b'TRP-GLY-GLY' b'VAL-CYS-GLY' b'VAL-GLY-CYS' b'TYR-GLY-GLY'
 b'TRP-ALA-GLY' b'TRP-CYS-GLY' b'THR-CYS-GLY' b'SER-CYS-GLY'
 b'TYR-ALA-GLY' b'TRP-GLY-ALA']
[b'TYR-GLY-GLY' b'TRP-GLY-GLY' b'TYR-CYS-GLY' b'TRP-ALA-GLY'
 b'TRP-CYS-GLY' b'TRP-GLY-ALA' b'TYR-GLY-CYS' b'TRP-GLY-CYS'
 b'TRP-HSE-GLY' b'THR-CYS-GLY']
[b'TYR-GLY-GLY' b'TRP-GLY-GLY' b'TYR-CYS-GLY' b'TYR-GLY-CYS'
 b'TRP-GLY-ALA' b'TRP-CYS-GLY' b'TRP-ALA-GLY' b'TRP-GLY-CYS'
 b'TRP-MET-GLY' b'TYR-CYS-CYS']
[b'TYR-TYR-PRO' b'TYR-PRO-PRO' b'TYR-TRP-PRO' b'TYR-TRP-VAL'
 b'TYR-TYR-MET' b'TYR-TYR-ILE' b'TYR-TYR-ARG' b'TYR-TYR-GLU'
 b'TYR-TYR-GLN' b'TYR-TRP-MET']
[b'TYR-TRP-VAL' b'TYR-TYR-VAL' b'TYR-TRP-PRO' b'TYR-VAL-PRO'
 b'TYR-TYR-PRO'